In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import base64

In [2]:
#..będziemy korzystać z trzech dataset 
main=pd.read_csv('pokemon_data.csv') #..informacje o Dual Type
evol=pd.read_csv('evolution_stage_map.csv') #..stopień ewolucji pokemona
paths=pd.read_csv('evolution_paths.csv') #..ścieżki ewolucji

In [3]:
#..dane z API i dane z pliku pokemon_data.csv różnią się nazwami pokemon - np. Nidoran♀ (female) i Nidoran-f. Poniższa funkcja normalizuje nazwy pokemon - usuwa z nich znaki jak niżej. Po zastosowaniu tej funkcji, a następnie merge how='inner' otrzymujemy 1025 wierszy, czyli tyle, ile miało być.
def normalize_name(name):
    name = str(name).lower()
    name = name.replace('♀ (female)', '-f')
    name = name.replace('♂ (male)', '-m')
    name = name.replace('é', 'e')   # np. Flabébé -> flabebe
    name = name.replace('.', '')    # np. Mr. Mime -> mr-mime
    name = name.replace("'", '')    # np. Farfetch'd -> farfetchd
    name = name.replace(':', '')    # np. Type: Null -> type-null
    name = name.replace(' ', '-')   
    return name


In [4]:
#------MERGING MAIN i EVOL--------
#..tworzymy klucz do łączenia w obu ramkach danych
main['merge_key'] = main['name'].apply(normalize_name)
evol['merge_key'] = evol['name'].apply(normalize_name)

#..łączymy po nowym kluczu
df = pd.merge(main, evol, on='merge_key', how='inner')
#..użyliśmy merge po kluczu, więc pandas stworzyło kolumny name_y i name_x - są takie same, więc jedną usuniemy i dla drugiej zmienimy nazwę. Ponadto usuniemy merge_key
df = df.drop(columns=['name_y'])
df = df.rename(columns={'name_x': 'name'})

#..upewniamy się, że Evolution_Stage to int, nie float
df['Evolution_Stage'] = df['Evolution_Stage'].astype(int)
df


,dexnum,name,generation,type1,type2,species,height,weight,ability1,ability2,...,base_exp,growth_rate,egg_group1,egg_group2,percent_male,percent_female,egg_cycles,special_group,merge_key,Evolution_Stage
0,1,Bulbasaur,1,Grass,Poison,Seed Pokémon,0.7,6.9,Overgrow,Chlorophyll,...,64,Medium Slow,Grass,Monster,87.5,12.5,20,Ordinary,bulbasaur,1
1,2,Ivysaur,1,Grass,Poison,Seed Pokémon,1.0,13.0,Overgrow,Chlorophyll,...,142,Medium Slow,Grass,Monster,87.5,12.5,20,Ordinary,ivysaur,2
2,3,Venusaur,1,Grass,Poison,Seed Pokémon,2.0,100.0,Overgrow,Chlorophyll,...,236,Medium Slow,Grass,Monster,87.5,12.5,20,Ordinary,venusaur,3
3,4,Charmander,1,Fire,NaN,Lizard Pokémon,0.6,8.5,Blaze,Solar Power,...,62,Medium Slow,Dragon,Monster,87.5,12.5,20,Ordinary,charmander,1
4,5,Charmeleon,1,Fire,NaN,Flame Pokémon,1.1,19.0,Blaze,Solar Power,...,142,Medium Slow,Dragon,Monster,87.5,12.5,20,Ordinary,charmeleon,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1020,1021,Raging Bolt,9,Electric,Dragon,Paradox Pokémon,5.2,480.0,Protosynthesis,NaN,...,—,Slow,Undiscovered,NaN,NaN,NaN,—,Ancient Paradox,raging-bolt,1
1021,1022,Iron Boulder,9,Rock,Psychic,Paradox Pokémon,1.5,162.5,Quark Drive,NaN,...,—,Slow,Undiscovered,NaN,NaN,NaN,—,Future Paradox,iron-boulder,1
1022,1023,Iron Crown,9,Steel,Psychic,Paradox Pokémon,1.6,156.0,Quark Drive,NaN,...,—,Slow,Undiscovered,NaN,NaN,NaN,—,Future Paradox,iron-crown,1
1023,1024,Terapagos,9,Normal,NaN,Tera Pokémon,0.2,6.5,Tera Shift,NaN,...,—,Slow,Undiscovered,NaN,50.0,50.0,—,Legendary,terapagos,1


In [5]:
#------MERGING DF i PATHS--------

#..przygotowywanie słownika - mapy typów. Tworzymy słownik zawierający znormalizowaną nazwę pokemona oraz to, czy jest Dual Type czy nie.
type_map = df.set_index(df['name'].apply(normalize_name))['type2'].notna().to_dict()

#..tworzymy klucze, przez które następnie będziemy mergować - normalizujemy imiona w każdym stage ewolucji
paths['Stage 1 Key'] = paths['Stage 1'].apply(lambda x: normalize_name(x) if pd.notna(x) else None)
paths['Stage 2 Key'] = paths['Stage 2'].apply(lambda x: normalize_name(x) if pd.notna(x) else None)
paths['Stage 3 Key'] = paths['Stage 3'].apply(lambda x: normalize_name(x) if pd.notna(x) else None)

#..wartości True/False dla wszystkich kategorii
paths['Can evolve? (S2)'] = paths['Stage 2'].notna()
paths['Can evolve? (S3)'] = paths['Stage 3'].notna()
#..tutaj brane są nazwy ewolucji z kolumny Stage X Key, a później sprawdzane, czy ma Dual Type za pomocą type_map utworzonego wcześniej
paths['S1: Dual Type?'] = paths['Stage 1 Key'].map(type_map)
paths['S2: Dual Type?'] = paths['Stage 2 Key'].map(type_map)
paths['S3: Dual Type?'] = paths['Stage 3 Key'].map(type_map)

#..w danych mamy np. 8 lini ewolucji Eevee. Dane muszą zostać zagregowane, by uniknąć powtórzeń. Grupujemy po 'Stage 1', .agg({ X : 'max'}) sprawia, że jeśli istnieje choć jedna linia, w której jest True, to w danej kolumnie przypisana będzie wartość True. 
paths_agg = paths.groupby('Stage 1').agg({
    'S1: Dual Type?':'max',
    'Can evolve? (S2)': 'max',
    'Can evolve? (S3)': 'max',
    'S2: Dual Type?': 'max',
    'S3: Dual Type?': 'max'
}).reset_index() #..groupby domyślnie ustawia Stage 1 jako indeks - resetujemy go

#..znowu normalizacja nazwy
paths_agg['merge_key'] = paths_agg['Stage 1'].apply(normalize_name)

#..finalny merge - musi być how='left', aby zachować Pokemony bez ewolucji. Zachowujemy tylko pokemony na Evolution_Stage==1
df_final = df[df['Evolution_Stage'] == 1].copy()
df_final['merge_key'] = df_final['name'].apply(normalize_name)

df_final = pd.merge(df_final, paths_agg.drop(columns='Stage 1'), on='merge_key', how='left')

In [6]:
#------DODAWANIE OBRAZKÓW - KONIECZNY KOD--------
#..Z uwagi na problemy z zapisywaniem plików przy pomocy Kaleido na Macbook M1 Pro, należało zmienić ścieżkę obrazków na format Base64. 
def get_image_source(path):
    #..plik czytamy binarnie - obrazki to ciąg zer i jedynek, bez "rb" python próbowałby odczytać dane jako litery
    with open(path, "rb") as image_file: 
        #..base64 zmienia dane binarne na kod ASCII, decode() zmienia bajty na string
        encoded_string = base64.b64encode(image_file.read()).decode()
    #..data URI scheme
    return "data:image/png;base64," + encoded_string

In [7]:
#------TWORZENIE WYKRESU W PLOTLY--------
#..definicja poziomów 
levels = [
    'S1: Dual Type?', 
    'Can evolve? (S2)', 
    'S2: Dual Type?', 
    'Can evolve? (S3)', 
    'S3: Dual Type?'
]

#..definicja kolorów dla True i False
color_map = {
    'True_node': "#3b8bba",    
    'False_node': "#d95f5f",   
    'True_link': "rgba(59, 139, 186, 0.4)", #..chcemy zmniejszyć opacity koloru, w plotly działa rgba() z czwartym argumentem będącym opacity (im większy tym ciemniejszy)
    'False_link': "rgba(217, 95, 95, 0.4)"  
}

#-----NODES-----
#..do diagramu Sankey potrzebne są tzw. nodes - pionowych pudełek kategorii. 
labels = []
colors = []
x_nodes = [] #..współrzędne x pudełka
y_nodes = [] #..współrzędne y pudełka
node_indices = {} #..słownik mapujący nazwę (np. "Dual type_True") na liczbę całkowitą, bo plotly łączy wstęgi między numerami indeksów, a nie nazwami
counter = 0 #..licznik, w której kategorii obecnie jesteśmy
for i, col in enumerate(levels):
    #..iterujemy tylko po True/False, a NaN są automatycznie pomijane
    for val in [True, False]:
        unique_id = f"{col}_{val}" #np. "Dual type_True"
        node_indices[unique_id] = counter
        
        #..nazwa kategorii z liczbą Pokémonów 
        count = len(df_final[df_final[col] == val]) #..ile Pokemonów w danej kolumnie ma daną wartość True/False
        label_text = f"<b>{col}<br>{val} (n={count})</b>" #..plotly ma htmlowe formatowanie - <> to pogrubienie
        labels.append(label_text)
        
        #..nadanie koloru
        colors.append(color_map['True_node'] if val else color_map['False_node'])
        
        #..pozycja x
        x_nodes.append(i / (len(levels) - 1))
        
        #..pozycja Y - wymuszenie True na górze. W plotly: 0.0 to góra, 1.0 to dół
        if i == 0:  #..pierwsza kategoria 
            y_nodes.append(0.01 if val else 0.99)
            
        elif i in [1]:  #..Can Evolve? (S2)
            if val: 
                y_nodes.append(0.1)  
            else:   
                y_nodes.append(0.99) 
        elif i in [3]: #..Can Evolve? (S3)
            if val: 
                y_nodes.append(0.1)  
            else:   
                y_nodes.append(0.7) 
        elif i in [2, 4]:  #..Is Dual Type? (S2 i S3)
            if val:
                y_nodes.append(0.05) 
            else:
                y_nodes.append(0.25) 

        counter += 1

#-----LINKS-----
#..utworzenie połączeń między kategoriami
source = [] #..indeksy, z których wychodzą połączenia
target = [] #..indeksy, do których trafiają
value = [] 
link_colors = []

for i in range(len(levels) - 1): #..iterowanie po parach kategorii, -1 bo ostatnia kategoria nie ma następcy z którym mogłaby być sparowana
    col_source = levels[i]
    col_target = levels[i+1]
    
    #..grupowanie danych po unikalnych ścieżkach; size() zlicza wystąpienia danej kombinacji, .reset_index(name='count') zamienia wynik w tabelę, gdzie kolumna count to liczba Pokémonów płynących daną trasą.
    flow = df_final.groupby([col_source, col_target]).size().reset_index(name='count') 

    #..wypakowujemy dane z zagregowanej tabeli flow i przygotowujemy do formatu, którego wymaga wykres Sankeya
    for _, row in flow.iterrows():
        src_val = row[col_source]
        tgt_val = row[col_target]
        count = row['count']
        
        #..jeśli przy Has Stage 2/3 otrzymał False, to kończy się dany przepływ
        if i in [1, 3] and src_val == False:
            continue
            
        #..używamy try/except - jeśli kolejny węzeł nie istnieje bo się skończył, to go pomijamy
        try:
            src_idx = node_indices[f"{col_source}_{src_val}"]
            tgt_idx = node_indices[f"{col_target}_{tgt_val}"]
        except KeyError:
            continue 
        
        source.append(src_idx)
        target.append(tgt_idx)
        value.append(count)
        link_colors.append(color_map['True_link'] if tgt_val else color_map['False_link'])


#---TWORZENIE WYKRESU---
fig = go.Figure(data=[go.Sankey(
    arrangement = "snap", #..rozdziela jakby okazało się, że podane przez nas współrzędne sprawiają, że kategorie na siebie nachodzą 
    node = dict(
        pad = 20, thickness = 20, line = dict(color = "black", width = 0.5),
        label = labels, color = colors, x = x_nodes, y = y_nodes
    ),
    link = dict(
        source = source, target = target, value = value, color = link_colors
    )
)])

#---DODANIE TYTUŁU---
fig.update_layout(
    title={
        #..ZMIANA - BRAK POGRUBIENIA TYTUŁU, ROZMIAR CZCIONKI 20->24
        'text': "Evolution and Dual Typing of Stage 1 Pokémon (N=541)", 
        'y': 0.95,           #..pozycja pionowa
        'x': 0.5,            #..pozycja pozioma - 0.5 to środek
        'font': dict(size=24)
    },
    font_size=14, 
    width=1600, height=800,
    plot_bgcolor='white',
    margin=dict(l=50, r=50, t=100, b=150)
)

#---DODANIE OBRAZKÓW---
#..dodanie poziomej linii na dole (oś X), aby Pokemony miały na czym stać
fig.add_shape(
    type="line",
    xref="paper", yref="paper",
    x0=0, y0=-0.18, x1=1, y1=-0.18, # Pozycja lekko poniżej wykresu
    line=dict(color="black", width=2),
    layer="below" 
)

fig.add_layout_image(
    dict(
        source=get_image_source('images/images/1.png'),
        xref="paper", yref="paper", #..dzięki temu plotly pozycjonuje względem całego obrazka, a nie jest ogramiczony do osi 
        x=0, y=0,  #..współrzędne na obrazku
        xanchor="left", yanchor="top",
        sizex=0.2, sizey=0.2, #..rozmiar obrazka
        layer="above" #..ustawienie obrazka na wierzchu
    )
)

fig.add_layout_image(
    dict(
        source=get_image_source('images/images/2.png'),
        xref="paper", yref="paper",
        x=0.47, y=0,  
        xanchor="left", yanchor="top",
        sizex=0.2, sizey=0.2,
        layer="above" 
    )
)

fig.add_layout_image(
    dict(
        source=get_image_source('images/images/3.png'),
        xref="paper", yref="paper",
        x=0.9, y=0.040, 
        xanchor="left", yanchor="top",
        sizex=0.3, sizey=0.3, 
        layer="above" 
    )
)


fig.write_image("sankey.png", scale=3) #..WYMAGANY KALEIDO (pip install kaleido) I GOOGLE CHROME/CHROMIUM! (jest szansa, że działa na Edge)
fig.show()

In [8]:
srednie_statystyki = df.groupby('Evolution_Stage')['total'].mean()

In [9]:
srednie_statystyki

Evolution_Stage
1    389.698706
2    453.311295
3    520.661157
Name: total, dtype: float64

In [11]:
df.groupby('generation')['total'].mean()

generation
1    407.642384
2    407.180000
3    403.725926
4    445.570093
5    425.756410
6    429.305556
7    449.409091
8    439.218750
9    457.391667
Name: total, dtype: float64